# RAG Hukum Ketenagakerjaan Indonesia — Pipeline Lengkap + Komparasi Model

Notebook ini menggabungkan seluruh pipeline RAG (Phase 5–11) dalam satu file yang siap dijalankan di **Google Colab**.

**Isi:**
- Phase 5-6: Embedding + ChromaDB + BM25
- Phase 7-8: Hybrid Retrieval + Neural Reranker
- Phase 9-10-11: Context Assembly + Generasi (Qwen3.5-9B) + Validasi
- Evaluasi batch otomatis + LLM-as-a-Judge
- Komparasi model: **Qwen3.5-9B** vs **GLM-4-9B-Chat** (`zai-org/glm-4-9b-chat`)

> Jalankan sel dari atas ke bawah secara berurutan.


## 0. Instalasi Package


In [ ]:
# Jalankan sekali di awal sesi. Runtime akan restart otomatis setelah install.
!pip install -q chromadb rank_bm25 sentence-transformers transformers accelerate \
    bitsandbytes pandas numpy matplotlib scikit-learn openpyxl tabulate tqdm datasets


## 1. Konfigurasi

Atur path ke data di sini. Dua opsi:
- **Google Drive** (persisten lintas sesi)
- **Colab local** (hilang saat sesi berakhir)


In [ ]:
import os

USE_DRIVE = True  # Ganti False jika tidak memakai Google Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = "/content/drive/MyDrive/RAG"
else:
    BASE_DIR = "/content/RAG"

os.makedirs(BASE_DIR + "/data", exist_ok=True)
os.makedirs(BASE_DIR + "/data/chroma_db", exist_ok=True)
os.makedirs(BASE_DIR + "/data/eval_outputs/plots", exist_ok=True)

DATA_PATH         = BASE_DIR + "/data/processed_chunks_ringan_pasal_chroma_ready.json"
CHROMA_DB_DIR     = BASE_DIR + "/data/chroma_db"
BM25_PATH         = BASE_DIR + "/data/bm25_index.pkl"
GROUND_TRUTH_CSV  = BASE_DIR + "/data/ground_truth_eval_20.csv"
OUTPUT_DIR        = BASE_DIR + "/data/eval_outputs"
PLOT_DIR          = OUTPUT_DIR + "/plots"

COLLECTION_NAME       = "hukum_ketenagakerjaan"
EMBEDDING_MODEL_NAME  = "intfloat/multilingual-e5-base"
RERANKER_MODEL_NAME   = "BAAI/bge-reranker-v2-m3"
SEMANTIC_MODEL_NAME   = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
QWEN_MODEL_ID         = "Qwen/Qwen3.5-9B"
GLM4_MODEL_ID         = "zai-org/glm-4-9b-chat"
MAX_NEW_TOKENS        = 1500

print("BASE_DIR:", BASE_DIR)
print("Data chunks:", DATA_PATH)
print("Ground truth:", GROUND_TRUTH_CSV)


## 2. Imports


In [ ]:
import gc, hashlib, json, math, os, pickle, re, time
from pathlib import Path
from typing import Dict, List, Any, Set

import chromadb
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## 3. Phase 5-6: Muat Data & Bangun Indeks (Embedding + ChromaDB + BM25)

Pastikan file `processed_chunks_ringan_pasal_chroma_ready.json` sudah ada di `data/`.
Jika ChromaDB dan BM25 sudah ada, sel ini akan **skip** rebuild secara otomatis.


In [ ]:
def ensure_unique_chunk_ids(chunks):
    seen, fixed = {}, 0
    for idx, chunk in enumerate(chunks):
        base_id = str(chunk.get("id") or f"chunk-{idx}")
        count = seen.get(base_id, 0)
        seen[base_id] = count + 1
        if count:
            seed = "::".join([
                base_id, str(idx),
                chunk.get("metadata", {}).get("source_file", ""),
                chunk.get("metadata", {}).get("pasal_id", ""),
                chunk.get("text", "")[:200],
            ])
            chunk["original_id"] = base_id
            chunk["id"] = hashlib.sha1(seed.encode()).hexdigest()
            fixed += 1
    if fixed:
        print(f"Fixed {fixed} duplicate chunk ids")
    return chunks


def normalize_metadata(chunk):
    meta = dict(chunk.get("metadata", {}))
    meta["chunk_id"]       = chunk.get("id", "")
    meta["citation_text"]  = chunk.get("citation_text", "")
    meta["year"]           = int(meta.get("year") or 0)
    meta["publication_year"] = int(meta.get("publication_year") or meta.get("year") or 0)
    safe = {}
    for k, v in meta.items():
        if v is None:                         safe[k] = ""
        elif isinstance(v, (str,int,float,bool)): safe[k] = v
        else:                                 safe[k] = json.dumps(v, ensure_ascii=False)
    return safe


def tokenize_bm25(text):
    return re.findall(r"[a-zA-Z0-9]+", str(text).lower())


def compact_citation(meta):
    reg_type = str(meta.get("regulation_type") or "Aturan").strip()
    nomor    = str(meta.get("nomor") or "").strip()
    year     = str(meta.get("publication_year") or meta.get("year") or "").strip()
    pasal    = str(meta.get("pasal_id") or "").strip()
    parts    = [reg_type]
    if nomor and nomor.lower() != "unknown": parts.append(f"No. {nomor}")
    if year and year != "0":                 parts.append(f"Tahun {year}")
    cit = " ".join(parts)
    return f"{cit}, {pasal}" if pasal else cit


# Load chunks
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"File tidak ditemukan: {DATA_PATH}\n"
        "Upload file processed_chunks_ringan_pasal_chroma_ready.json ke folder data/"
    )
chunks = json.loads(open(DATA_PATH, encoding="utf-8").read())
chunks = ensure_unique_chunk_ids(chunks)
print(f"Loaded {len(chunks)} chunks")
print("Sample citation:", chunks[0]["citation_text"])


In [ ]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=DEVICE)
chroma_client   = chromadb.PersistentClient(path=CHROMA_DB_DIR)

FORCE_REBUILD = False  # Ubah True hanya jika chunk berubah
expected_count = len(chunks)
try:
    collection    = chroma_client.get_collection(COLLECTION_NAME)
    current_count = collection.count()
    print(f"Existing Chroma count: {current_count}")
except Exception:
    collection    = None
    current_count = 0

if collection is None or current_count != expected_count or FORCE_REBUILD:
    if collection is not None:
        chroma_client.delete_collection(COLLECTION_NAME)
        print("Deleted old collection.")
    collection = chroma_client.get_or_create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"},
    )
    batch_size = 256 if DEVICE == "cuda" else 32
    for start in tqdm(range(0, expected_count, batch_size), desc="Embedding+storing"):
        batch = chunks[start:start+batch_size]
        ids        = [c["id"] for c in batch]
        passages   = ["passage: " + c["embedding_text"] for c in batch]
        embeddings = embedding_model.encode(passages, batch_size=batch_size,
                         convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False).tolist()
        documents  = [c["display_text"] for c in batch]
        metadatas  = [normalize_metadata(c) for c in batch]
        collection.add(ids=ids, embeddings=embeddings, documents=documents, metadatas=metadatas)
    print("ChromaDB built:", collection.count())
else:
    print("ChromaDB OK — skip rebuild.")

# BM25
if os.path.exists(BM25_PATH):
    with open(BM25_PATH, "rb") as f:
        payload  = pickle.load(f)
    bm25     = payload["bm25"]
    bm25_ids = payload["ids"]
    print(f"BM25 loaded: {len(bm25_ids)} docs")
else:
    corpus   = [tokenize_bm25(c["display_text"] + " " + c["embedding_text"]) for c in chunks]
    bm25     = BM25Okapi(corpus)
    bm25_ids = [c["id"] for c in chunks]
    with open(BM25_PATH, "wb") as f:
        pickle.dump({"bm25": bm25, "ids": bm25_ids}, f)
    print(f"BM25 built & saved: {len(bm25_ids)} docs")


## 4. Phase 7: Fungsi Retrieval (Dense + BM25 + RRF)


In [ ]:
KETENAGAKERJAAN_POSITIVE = {
    "ketenagakerjaan", "tenaga kerja", "pekerja", "buruh", "hubungan kerja",
    "perjanjian kerja", "pemutusan hubungan kerja", "phk", "pesangon",
    "upah", "pengupahan", "pensiun", "jaminan sosial", "jaminan kerja",
    "alih daya", "outsourcing", "pkwt", "pkwtt", "serikat pekerja",
    "pengusaha", "perusahaan alih daya", "perlindungan pekerja",
    "waktu kerja", "cuti", "k3", "keselamatan kerja", "cipta kerja",
}
KETENAGAKERJAAN_NEGATIVE = {
    "perizinan berusaha", "oss", "risiko usaha", "izin usaha", "nib",
    "investasi", "badan usaha", "penyelenggaraan usaha", "sistem perizinan",
    "rba", "risk based approach", "sektor usaha", "kbli",
    "penyelenggaraan pemerintahan", "administrasi pemerintahan",
    "sop administrasi", "pelayanan publik",
}
KNOWN_REGS = {
    ("pp","35"),("pp","36"),("pp","34"),("pp","45"),("pp","37"),
    ("uu","13"),("uu","6"),("uu","24"),("uu","1"),("uu","21"),("uu","2"),
    ("permen","5"),("permen","6"),("permen","21"),("pp","44"),("pp","10"),
}

def norm_reg(v):
    v = str(v or "").lower()
    if "undang" in v or v == "uu":   return "uu"
    if "pemerintah" in v or v == "pp": return "pp"
    if "presiden" in v or "perpres" in v: return "perpres"
    if "menteri" in v or "permen" in v:   return "permen"
    return v

def is_ketenaga(meta):
    haystack = " ".join([str(meta.get(k,"") or "").lower()
                          for k in ("tentang","bab_title","bagian_title")])
    if any(neg in haystack for neg in KETENAGAKERJAAN_NEGATIVE): return False
    rt  = norm_reg(meta.get("regulation_type",""))
    nom = str(meta.get("nomor","") or "").strip()
    if (rt, nom) in KNOWN_REGS: return True
    if any(pos in haystack for pos in KETENAGAKERJAAN_POSITIVE): return True
    return not haystack.strip()


def lex_score(hit):
    meta  = hit.get("metadata", {})
    score = float(hit.get("rrf_score", 0.0))
    try:    year = int(meta.get("publication_year") or meta.get("year") or 0)
    except: year = 0
    try:    hier = int(meta.get("regulation_hierarchy") or 99)
    except: hier = 99
    if str(meta.get("active_status","")).lower() == "berlaku": score += 0.030
    if is_ketenaga(meta): score += min(max(year-2000,0),40)*0.001
    else:                 score -= 0.100
    score += max(0, 6-hier)*0.003
    if meta.get("quality_status") == "needs_review": score -= 0.010
    return score


def dense_search(query, fetch_k=60):
    q_emb = embedding_model.encode(["query: "+query], convert_to_numpy=True,
                                    normalize_embeddings=True, show_progress_bar=False)[0].tolist()
    res = collection.query(query_embeddings=[q_emb], n_results=fetch_k,
                            include=["documents","metadatas","distances"])
    return [{"id": did, "text": doc, "metadata": meta, "dense_distance": float(d), "source": "dense"}
            for did, doc, meta, d in zip(res["ids"][0], res["documents"][0], res["metadatas"][0], res["distances"][0])]


def bm25_search(query, fetch_k=60):
    if bm25 is None: return []
    scores = bm25.get_scores(tokenize_bm25(query))
    order  = np.argsort(scores)[::-1][:fetch_k]
    ids    = [bm25_ids[i] for i in order if scores[i] > 0]
    if not ids: return []
    got    = collection.get(ids=ids, include=["documents","metadatas"])
    lookup = {did: (doc, meta) for did, doc, meta in zip(got["ids"], got["documents"], got["metadatas"])}
    hits   = []
    for i in order:
        did = bm25_ids[i]
        if scores[i] <= 0 or did not in lookup: continue
        doc, meta = lookup[did]
        hits.append({"id": did, "text": doc, "metadata": meta or {}, "bm25_score": float(scores[i]), "source": "bm25"})
    return hits


def rrf_fuse(result_sets, weights=None, k=60):
    weights = weights or [1.0]*len(result_sets)
    fused   = {}
    for hits, w in zip(result_sets, weights):
        for rank, hit in enumerate(hits, 1):
            item = fused.setdefault(hit["id"], {"score":0.0, "hit":hit})
            item["score"] += w/(k+rank)
    out = []
    for item in sorted(fused.values(), key=lambda x: x["score"], reverse=True):
        hit = item["hit"]; hit["rrf_score"] = item["score"]; out.append(hit)
    return out


def dedupe_legal_hits(hits, k=8):
    ranked = sorted(hits, key=lex_score, reverse=True)
    seen, out = set(), []
    for hit in ranked:
        meta = hit.get("metadata", {})
        if not is_ketenaga(meta): continue
        key = (meta.get("source_file",""), meta.get("pasal_id",""),
               meta.get("chunk_kind",""), meta.get("chunk_index",""))
        if key in seen: continue
        seen.add(key)
        hit["final_score"] = lex_score(hit)
        out.append(hit)
        if len(out) >= k: break
    return out


def retrieve_docs(query, k=8, fetch_k=40):
    dense = dense_search(query, fetch_k=fetch_k)
    sparse = bm25_search(query, fetch_k=fetch_k)
    fused  = rrf_fuse([dense, sparse], weights=[1.0,1.0]) if sparse else dense
    return dedupe_legal_hits(fused, k=k)

print("Retrieval functions ready.")


## 5. Phase 8-9: Reranker + Context Assembly + Query Expansion


In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(RERANKER_MODEL_NAME, device=DEVICE)
print("Reranker ready:", RERANKER_MODEL_NAME)


def rerank_docs(query, docs, k=8):
    if not docs: return docs[:k]
    scores = reranker.predict([(query, d["text"]) for d in docs])
    for d, s in zip(docs, scores): d["rerank_score"] = float(s)
    return sorted(docs, key=lambda x: x.get("rerank_score",0.0), reverse=True)[:k]


In [ ]:
LEGAL_EXPANSIONS = [
    {"triggers":["kompensasi","uang kompensasi","kontraknya berakhir"],
     "expansion":"PKWT berakhir jangka waktu uang kompensasi Pasal 15 Pasal 16 masa kerja dibagi 12 dikali 1 bulan upah"},
    {"triggers":["bahasa","huruf latin","penafsiran"],
     "expansion":"PKWT wajib tertulis huruf latin bahasa Indonesia naskah bahasa Indonesia berlaku Pasal 13"},
    {"triggers":["lembur","melebihi waktu kerja"],
     "expansion":"lembur perintah pengusaha persetujuan pekerja tertulis media digital Pasal 28"},
    {"triggers":["uang penggantian hak","komponen uang","phk secara umum"],
     "expansion":"Pasal 40 pesangon penghargaan masa kerja cuti tahunan ongkos pulang pekerja keluarga"},
    {"triggers":["bipartit","mediasi","sengketa","pekerja menolak"],
     "expansion":"bipartit 30 hari dicatatkan instansi mediasi konsiliasi PHI UU 13 Pasal 151 UU 2 Pasal 3"},
    {"triggers":["jkp","jaminan kehilangan pekerjaan"],
     "expansion":"JKP uang tunai 45 persen 3 bulan 25 persen 3 bulan akses informasi pelatihan PP 37 2021"},
    {"triggers":["pelanggaran berat","mendesak","bersifat mendesak"],
     "expansion":"pelanggaran bersifat mendesak uang pisah uang penggantian hak tidak mendapat pesangon"},
    {"triggers":["surat peringatan","sp pertama","sp kedua","sp ketiga"],
     "expansion":"pelanggaran ketentuan SP 1 2 3 pesangon 0.5 penghargaan masa kerja uang penggantian hak"},
    {"triggers":["pesangon","phk","pemutusan hubungan kerja"],
     "expansion":"pemutusan hubungan kerja pesangon penghargaan masa kerja uang penggantian hak"},
    {"triggers":["pkwt","kontrak"],
     "expansion":"perjanjian kerja waktu tertentu kompensasi jangka waktu perpanjangan pembaruan"},
    {"triggers":["alih daya","outsourcing"],
     "expansion":"alih daya perusahaan alih daya hubungan kerja perlindungan upah"},
    {"triggers":["upah","gaji","tunjangan"],
     "expansion":"upah gaji tunjangan struktur skala upah minimum pembayaran"},
]

STOPWORDS = {
    "yang","dan","atau","karena","dengan","untuk","pada","dalam","jika","maka",
    "berapa","apakah","bagaimana","pekerja","buruh","pengusaha","perusahaan",
    "hak","di","ke","dari","atas","nya","ia","ini","itu","ada","dapat","bisa",
}

def expand_query(query):
    q, exps = query.lower(), []
    for rule in LEGAL_EXPANSIONS:
        if any(t in q for t in rule["triggers"]):
            exps.append(rule["expansion"])
    return re.sub(r"\s+"," "," ".join([query]+exps)).strip()


def query_terms(query):
    return {t for t in re.findall(r"[a-zA-Z0-9]+",query.lower())
            if len(t)>2 and t not in STOPWORDS}


def doc_relevance(query, doc):
    terms   = query_terms(expand_query(query))
    if not terms: return 1.0
    meta    = doc.get("metadata", {})
    haystack = " ".join([str(doc.get("text","")),
                          str(meta.get("citation_text","")),
                          str(meta.get("pasal_id","")),
                          str(meta.get("bab_title","")),]).lower()
    score = sum(1 for t in terms if t in haystack) / len(terms)
    orig  = {t for t in query_terms(query) if t not in {"pesangon","phk","pemutusan","kerja"}}
    if orig and not any(t in haystack for t in orig): score -= 0.25
    if "final_score" in doc: score += min(float(doc["final_score"]),1.0)*0.05
    return score


def filter_context(query, docs, max_docs=8):
    orig = {t for t in query_terms(query) if t not in {"pesangon","phk","pemutusan","kerja"}}
    scored = []
    for d in docs:
        meta = d.get("metadata",{})
        hs   = " ".join([str(d.get("text","")),str(meta.get("bab_title","")),]).lower()
        if orig and not any(t in hs for t in orig): continue
        s = doc_relevance(query, d)
        if s > 0: d["ctx_score"]=s; scored.append(d)
    return sorted(scored, key=lambda x:x.get("ctx_score",0), reverse=True)[:max_docs] if scored else docs[:1]


def retrieve_context(query, k=8, fetch_k=80):
    cands = {}
    for q in [query, expand_query(query)]:
        for hit in retrieve_docs(q, k=max(k*8,30), fetch_k=fetch_k):
            ex = cands.get(hit["id"])
            if ex is None or hit.get("final_score",0)>ex.get("final_score",0):
                cands[hit["id"]] = hit
    # Guaranteed retrieval: inject pasal dari QUERY_ARTICLE_HINTS langsung
    for hinted in fetch_hinted_chunks(query):
        if hinted["id"] not in cands:
            cands[hinted["id"]] = hinted
    reranked = rerank_docs(expand_query(query), list(cands.values()), k=max(k*8,30))
    return filter_context(query, reranked, max_docs=k)


def build_context(docs, max_docs=None):
    parts=[]
    for i, d in enumerate((docs[:max_docs] if max_docs else docs), 1):
        cit  = compact_citation(d.get("metadata",{}))
        text = re.sub(r"\s+"," ",d.get("text","")).strip()[:2500]
        parts.append(f"SUMBER HUKUM {i}: {cit}\n{text}")
    return "\n\n".join(parts)

print("Context assembly ready.")


In [ ]:
# QUERY_ARTICLE_HINTS: peta query -> pasal yang PASTI harus diambil
# Digunakan oleh query_article_hint_score (boost skor) DAN
# fetch_hinted_chunks (guaranteed retrieval langsung dari chunk list).
QUERY_ARTICLE_HINTS = [
    {"triggers":["kompensasi","uang kompensasi","kontraknya berakhir"],
     "targets":[("pp","35","Pasal 15"),("pp","35","Pasal 16")]},
    {"triggers":["bahasa","huruf latin","penafsiran","warga negara indonesia"],
     "targets":[("pp","35","Pasal 13")]},
    {"triggers":["lembur","melebihi waktu kerja"],
     "targets":[("pp","35","Pasal 28")]},
    {"triggers":["uang penggantian hak","komponen uang","phk secara umum"],
     "targets":[("pp","35","Pasal 40")]},
    {"triggers":["surat peringatan","sp pertama","sp kedua","sp ketiga"],
     "targets":[("pp","35","Pasal 52")]},
    {"triggers":["bipartit","mediasi","pekerja menolak","sengketa"],
     "targets":[("uu","13","Pasal 151"),("uu","2","Pasal 3"),("uu","2","Pasal 4"),("pp","35","Pasal 52")]},
    {"triggers":["jkp","jaminan kehilangan pekerjaan","kehilangan pekerjaan"],
     "targets":[("pp","37","Pasal 18"),("pp","37","Pasal 21"),("pp","37","Pasal 25"),("pp","35","Pasal 40")]},
    {"triggers":["pemagangan","disabilitas","penyandang disabilitas"],
     "targets":[("permen","6","Pasal 13"),("permen","6","Pasal 16"),("permen","6","Pasal 24")]},
    {"triggers":["narkotika","psikotropika","jkk","jkm"],
     "targets":[("pp","44","Pasal 27"),("pp","44","Pasal 35")]},
    {"triggers":["bp2mi","pekerja migran","pmi","visa kerja","dokumen wajib pmi"],
     "targets":[("pp","10","Pasal 2"),("pp","10","Pasal 3"),("pp","10","Pasal 5"),("uu","18","Pasal 13"),("uu","18","Pasal 14")]},
    {"triggers":["pelanggaran berat","mendesak","bersifat mendesak"],
     "targets":[("pp","35","Pasal 52")]},
]


def query_article_hint_score(query: str, doc: dict) -> float:
    q    = query.lower()
    meta = doc.get("metadata", {})
    rt   = norm_reg(meta.get("regulation_type", ""))
    nom  = str(meta.get("nomor", "") or "").strip()
    pid  = str(meta.get("pasal_id", "") or "").strip().lower()
    score = 0.0
    for rule in QUERY_ARTICLE_HINTS:
        if not any(t in q for t in rule["triggers"]):
            continue
        for tt, tn, tp in rule["targets"]:
            if rt == tt and nom == tn and pid == tp.lower():
                score += 0.80
    return score


def fetch_hinted_chunks(query: str) -> list:
    """Guaranteed retrieval: tarik langsung chunk pasal dari QUERY_ARTICLE_HINTS.
    Memastikan pasal kritis (Pasal 13, 16, 28, PP 37 Pasal 18, dll)
    selalu ada di candidate pool meski dense/BM25 tidak menariknya.
    """
    q = query.lower()
    result, seen = [], set()
    for rule in QUERY_ARTICLE_HINTS:
        if not any(t in q for t in rule["triggers"]):
            continue
        for target_type, target_nomor, target_pasal in rule["targets"]:
            for chunk in chunks:
                meta = chunk.get("metadata", {})
                if (norm_reg(meta.get("regulation_type","")) == target_type
                        and str(meta.get("nomor","") or "").strip() == target_nomor
                        and str(meta.get("pasal_id","") or "").strip().lower() == target_pasal.lower()):
                    cid = chunk["id"]
                    if cid in seen:
                        continue
                    seen.add(cid)
                    result.append({
                        "id": cid,
                        "text": chunk.get("display_text") or chunk.get("text", ""),
                        "metadata": {k: (v if v is not None else "") for k, v in chunk.get("metadata", {}).items()},
                        "rrf_score": 0.60,
                        "final_score": 0.60,
                        "source": "hinted_article",
                    })
    return result


print("Article hints ready:", len(QUERY_ARTICLE_HINTS), "rules")


## 6. Muat Model Qwen3.5-9B


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

qwen_quant = None
if torch.cuda.is_available():
    qwen_quant = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

qwen_tokenizer = AutoTokenizer.from_pretrained(
    QWEN_MODEL_ID, trust_remote_code=True, padding_side="left"
)
if qwen_tokenizer.pad_token is None:
    qwen_tokenizer.pad_token = qwen_tokenizer.eos_token

qwen_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    quantization_config=qwen_quant,
    trust_remote_code=True,
)
qwen_model.eval()
print("Qwen ready:", QWEN_MODEL_ID)
if torch.cuda.is_available():
    print(f"VRAM allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")


## 7. Fungsi Generasi & Validasi (Phase 10-11)


In [ ]:
TYPO_PATTERNS = [
    r"[\u0400-\u04FF\u0600-\u06FF]",
    r"pemrintah|tahu\s+20\d{2}|peringatann|berturut[- ]?tutur|mendesar|presangon",
]
TYPO_FIXES = [
    (r"\bPemrintah\b","Pemerintah"),(r"\bTahu\s+(20\d{2})\b",r"Tahun \1"),
    (r"\bperingatann\b","peringatan"),(r"\bberturut[- ]?tutur\b","berturut-turut"),
    (r"\bmendesar\b","mendesak"),(r"\bpresangon\b","pesangon"),
]

def strip_thinking(text):
    return re.sub(r"<think>[\s\S]*?</think>","",text,flags=re.IGNORECASE).strip()

def sanitize(text):
    text = strip_thinking(text)
    text = re.sub(r"```.*?```","",text,flags=re.DOTALL)
    text = re.sub(r"\*\*(.*?)\*\*",r"\1",text)
    text = re.sub(r"\*(.*?)\*",r"\1",text)
    text = re.sub(r"^#{1,6}\s*","",text,flags=re.MULTILINE)
    text = re.sub(r"\s*\[(?:R\d+(?:\s*,\s*)?)+\]","",text,flags=re.IGNORECASE)
    for pat, rep in TYPO_FIXES: text = re.sub(pat,rep,text,flags=re.IGNORECASE)
    return re.sub(r"\n{3,}","\n\n",text).strip()

def has_quality_issue(text):
    if len(text.strip()) < 20: return True
    return any(re.search(p,text,flags=re.IGNORECASE) for p in TYPO_PATTERNS)

print("Util functions ready.")


In [ ]:
SYSTEM_PROMPT = "Kamu adalah pakar hukum ketenagakerjaan Indonesia yang sangat teliti.\nSaat membaca dokumen hukum, kamu harus memahami bahwa setiap AYAT memiliki kondisi (syarat) dan konsekuensi yang BERBEDA.\n\nATURAN UTAMA:\n1. JAWAB HANYA berdasarkan KONTEKS. DILARANG mengarang atau memakai pengetahuan luar.\n2. Sebutkan sumber hukumnya secara natural, misal: Peraturan Pemerintah No. 35 Tahun 2021, Pasal 52 ayat (2).\n3. DILARANG memakai kode [R1], [R2], SUMBER HUKUM 1, atau ID internal lain di jawaban.\n4. DILARANG menulis daftar referensi di dalam jawaban.\n5. Petakan istilah awam ke istilah resmi (pelanggaran berat = pelanggaran bersifat mendesak, kontrak = PKWT, dipecat = PHK).\n6. Hanya gunakan 'Maaf, informasi tidak tersedia' jika benar-benar tidak ada dokumen relevan.\n7. SPESIFIK: Sebutkan Pasal dan AYAT-nya dengan presisi. Jangan mencampuradukkan konsekuensi antar ayat!\n8. KOMPREHENSIF: Sebutkan semua komponen hak/kewajiban/tata cara lengkap.\n9. Bahasa Indonesia rapi, baku, tanpa typo, tanpa markdown.\n10. ATURAN KRITIS — Pasal 52 PP 35/2021:\n    Ayat (1): PHK setelah SP1,SP2,SP3 => pesangon 0.5x, UPMK 1x, UPH.\n    Ayat (2): PHK pelanggaran mendesak (tanpa SP) => TIDAK BERHAK pesangon & UPMK, hanya UPH + Uang Pisah.\n    DILARANG KERAS mencampur kedua kondisi ini.\n11. PKWT kompensasi PP 35/2021 Pasal 16 - RUMUS MUTLAK: masa_kerja/12 x 1 bulan upah. Contoh WAJIB: 6 bulan = 6/12 x 1 bulan upah = 0,5 bulan upah. DILARANG tulis 6 bulan upah.\n12. Bahasa PKWT Pasal 13 - TIGA SYARAT: (1) TERTULIS, (2) HURUF LATIN, (3) BAHASA INDONESIA. Jika beda tafsir dengan versi asing: naskah bahasa Indonesia berlaku.\n13. Lembur Pasal 28 - DUA SYARAT KUMULATIF: (1) PERINTAH TERTULIS dari pengusaha, DAN (2) PERSETUJUAN pekerja/buruh tertulis/media digital. Sebutkan KEDUANYA.\n14. UPH Pasal 40 ayat (4) - TIGA KOMPONEN: (1) cuti tahunan belum diambil/gugur, (2) ongkos pulang pekerja+keluarga, (3) hal lain dalam PK/PP/PKB.\n15. JKP PP 37/2021: uang tunai max 6 bulan - 45 persen x upah bln 1-3; 25 persen x upah bln 4-6.\n16. Sengketa PHK - 6 LANGKAH: cegah PHK, pemberitahuan, bipartit 30 hari, catat instansi, mediasi/konsiliasi, PHI.\n17. Jika soal tentang pemagangan/K3/PMI/disabilitas: JANGAN mapping ke pelanggaran berat/PHK/pesangon.\n18. Multi-poin: jawab SEMUA poin sampai tuntas, DILARANG terpotong di tengah.\n19. PMI - dokumen wajib: sertifikat kompetensi, surat sehat, paspor, visa kerja, perjanjian penempatan, perjanjian kerja, dokumen sesuai negara tujuan (PP 10/2020, UU 18/2017).\n"

def build_messages(question, docs, max_docs=8):
    context = build_context(docs, max_docs=max_docs)
    return [
        {"role":"system","content":SYSTEM_PROMPT},
        {"role":"user","content":f"KONTEKS REFERENSI HUKUM:\n{context}\n\nPERTANYAAN PENGGUNA:\n{question}"},
    ]


def generate_qwen(messages, max_new_tokens=MAX_NEW_TOKENS):
    if hasattr(qwen_tokenizer,"apply_chat_template"):
        try:    prompt = qwen_tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True,enable_thinking=False)
        except: prompt = qwen_tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    else:
        prompt = "\n".join(f"{m['role'].upper()}: {m['content']}" for m in messages)+"\nASSISTANT:"
    inputs = qwen_tokenizer(prompt,return_tensors="pt",padding=True,truncation=True,max_length=12000).to(qwen_model.device)
    with torch.no_grad():
        out = qwen_model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=True, temperature=0.04, top_p=0.9,
            eos_token_id=getattr(qwen_tokenizer,"eos_token_id",None),
            pad_token_id=getattr(qwen_tokenizer,"eos_token_id",None),
        )
    return qwen_tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],skip_special_tokens=True).strip()


def run_rag_qwen(question, k=10, max_docs=10):
    t0   = time.time()
    docs = retrieve_context(question, k=k)
    msgs = build_messages(question, docs, max_docs=max_docs)
    ans  = sanitize(generate_qwen(msgs))
    latency = time.time()-t0
    refs = []
    for i,d in enumerate(docs,1):
        m = d.get("metadata",{})
        refs.append({"rank":i,"chunk_id":d.get("id",""),"reference":compact_citation(m),
                     "source_file":m.get("source_file",""),"pasal_id":m.get("pasal_id",""),
                     "rrf_score":d.get("rrf_score"),"rerank_score":d.get("rerank_score"),
                     "text_preview":re.sub(r"\s+"," ",d.get("text","")).strip()[:500]})
    return {"question":question,"answer":ans,"references":refs,"latency_seconds":latency}

print("Qwen generation functions ready.")


## 8. Uji Cepat Pipeline Qwen


In [ ]:
test_result = run_rag_qwen("Jika pekerja di-PHK karena pelanggaran berat, berapa pesangonnya?", k=4)
print("PERTANYAAN:", test_result["question"])
print("\nJAWABAN:")
print(test_result["answer"])
print("\nREFERENSI DIAMBIL:")
for r in test_result["references"]:
    print(f"  [{r['rank']}] {r['reference']}")
print(f"\nLatency: {test_result['latency_seconds']:.2f}s")


## 9. Evaluasi Batch — Model Qwen


In [ ]:
import os
if not os.path.exists(GROUND_TRUTH_CSV):
    raise FileNotFoundError(f"Ground truth tidak ditemukan: {GROUND_TRUTH_CSV}")
gt = pd.read_csv(GROUND_TRUTH_CSV)
required_cols = {"id","topic","question","expected_answer","expected_keywords",
                  "expected_articles","expected_law_numbers","expected_citations"}
missing = required_cols - set(gt.columns)
if missing: raise ValueError(f"Kolom hilang: {sorted(missing)}")
print("Ground truth:", len(gt), "pertanyaan")
gt.head()


In [ ]:
print("Batch inference Qwen...")
qwen_results = []
all_docs_json = []  # Simpan retrieved docs untuk dipakai ulang oleh GLM-4

for row in tqdm(gt.to_dict("records"), desc="Qwen inference"):
    out = run_rag_qwen(row["question"], k=10, max_docs=8)
    refs_json = json.dumps(out["references"], ensure_ascii=False)
    all_docs_json.append(refs_json)
    qwen_results.append({
        **row,
        "model_id": QWEN_MODEL_ID,
        "answer": out["answer"],
        "latency_seconds": out["latency_seconds"],
        "retrieved_references_json": refs_json,
        "retrieved_references_text": " || ".join([r["reference"] for r in out["references"]]),
        "top1_reference": out["references"][0]["reference"] if out["references"] else "",
        "retrieved_count": len(out["references"]),
    })

df_qwen = pd.DataFrame(qwen_results)
print("Selesai:", len(df_qwen), "pertanyaan")


In [ ]:
def norm(s): return re.sub(r"\s+"," ",str(s).lower()).strip()
def compact_str(s): return re.sub(r"[^0-9a-z]+","",norm(s))
def split_targets(s): return [x.strip() for x in str(s).split(";") if x.strip() and x.strip().lower()!="nan"]

def keyword_hits(answer, keywords):
    ks, a, ac = split_targets(keywords), norm(answer), compact_str(answer)
    hits = [k for k in ks if norm(k) in a or compact_str(k) in ac]
    return {"hits":hits,"missing":[k for k in ks if k not in hits],
            "score":len(hits)/len(ks) if ks else 0.0}

def contains_target(blob, target):
    b, bc = norm(blob), compact_str(blob)
    for v in [norm(target), compact_str(target)]:
        if v and (v in b or v in bc): return True
    return False

def contains_any(blob, vals): return int(any(contains_target(blob,v) for v in vals))
def target_coverage(blob, vals):
    ts = [v for v in vals if str(v).strip()]
    return sum(1 for v in ts if contains_target(blob,v))/len(ts) if ts else 0.0

def parse_refs(js):
    try: r=json.loads(js); return r if isinstance(r,list) else []
    except: return []

def target_rank(row, targets):
    for r in parse_refs(row.get("retrieved_references_json","[]")):
        blob = " ".join([str(r.get(k,"")) for k in ["reference","pasal_id","source_file","text_preview"]])
        if contains_any(blob, targets): return int(r.get("rank",999))
    return 999

def reciprocal_rank(rank): return 0.0 if rank>=999 else 1.0/rank

def precision_at_k(row, targets, k=3):
    refs=parse_refs(row.get("retrieved_references_json","[]"))[:k]
    if not refs: return 0.0
    hits=sum(contains_any(" ".join([str(r.get(f,"")) for f in ["reference","pasal_id","source_file","text_preview"]]),targets) for r in refs)
    return hits/len(refs)

def answer_length_score(answer):
    w=len(str(answer).split())
    if w<25: return w/25
    if w<=220: return 1.0
    return max(0.3, 1.0-((w-220)/300))

WEIGHTS = {
    "llm_judge_score":0.20,"semantic_similarity":0.15,"keyword_coverage":0.15,
    "retrieval_citation_coverage":0.15,"retrieval_article_hit":0.10,"retrieval_law_hit":0.05,
    "answer_citation_hit":0.10,"article_hit_at_3":0.05,"article_mrr":0.03,
    "precision_at_3":0.01,"answer_length_score":0.01,
}
print("Metric functions ready.")


In [ ]:
def compute_metrics(df):
    kw = df.apply(lambda r: keyword_hits(r.get("answer",""), r.get("expected_keywords","")), axis=1)
    df["keyword_hits"]    = kw.apply(lambda x: "; ".join(x["hits"]))
    df["keyword_missing"] = kw.apply(lambda x: "; ".join(x["missing"]))
    df["keyword_coverage"]= kw.apply(lambda x: x["score"])
    df["expected_article_rank"] = df.apply(lambda r: target_rank(r, split_targets(r.get("expected_articles",""))), axis=1)
    df["expected_law_rank"]     = df.apply(lambda r: target_rank(r, split_targets(r.get("expected_law_numbers",""))), axis=1)
    df["retrieval_citation_coverage"] = df.apply(lambda r: target_coverage(r.get("retrieved_references_text",""), split_targets(r.get("expected_citations",""))), axis=1)
    df["retrieval_citation_hit"]= (df["retrieval_citation_coverage"]>0).astype(int)
    df["retrieval_article_hit"] = (df["expected_article_rank"]<999).astype(int)
    df["retrieval_law_hit"]     = (df["expected_law_rank"]<999).astype(int)
    df["article_hit_at_3"]      = (df["expected_article_rank"]<=3).astype(int)
    df["article_hit_at_8"]      = (df["expected_article_rank"]<=8).astype(int)
    df["article_mrr"]   = df["expected_article_rank"].apply(reciprocal_rank)
    df["law_mrr"]       = df["expected_law_rank"].apply(reciprocal_rank)
    df["precision_at_3"]= df.apply(lambda r: precision_at_k(r, split_targets(str(r.get("expected_articles",""))+";"+str(r.get("expected_law_numbers",""))), k=3), axis=1)
    df["answer_citation_hit"] = df.apply(lambda r: contains_any(r.get("answer",""), split_targets(str(r.get("expected_citations",""))+";"+str(r.get("expected_articles",""))+";"+str(r.get("expected_law_numbers","")))), axis=1)
    df["answer_word_count"]   = df["answer"].fillna("").apply(lambda x: len(str(x).split()))
    df["answer_length_score"] = df["answer"].apply(answer_length_score)
    return df

df_qwen = compute_metrics(df_qwen)
print("Metrik dihitung.")
df_qwen[["id","keyword_coverage","retrieval_citation_coverage","retrieval_article_hit","article_mrr"]].head()


In [ ]:
sem_model = SentenceTransformer(SEMANTIC_MODEL_NAME, device=DEVICE)

def compute_semantic_sim(df):
    exp = df["expected_answer"].fillna("").tolist()
    ans = df["answer"].fillna("").tolist()
    e   = sem_model.encode(exp, normalize_embeddings=True, show_progress_bar=True)
    a   = sem_model.encode(ans, normalize_embeddings=True, show_progress_bar=True)
    df["semantic_similarity"] = [float(np.dot(ai, ei)) for ai, ei in zip(a, e)]
    return df

df_qwen = compute_semantic_sim(df_qwen)
print("Semantic similarity computed.")


In [ ]:
def llm_judge(question, context, answer, expected, gen_fn):
    prompt = (
        f"Kamu adalah instruktur hukum ahli. Berikan nilai kualitas jawaban.\n"
        f"Nilai: angka 0-100.\n\n"
        f"KRITERIA: Akurasi hukum, kejelasan, dukungan referensi.\n\n"
        f"REFERENSI:\n{context}\n\nKUNCI JAWABAN:\n{expected}\n\nJAWABAN:\n{answer}\n\nNILAI:"
    )
    try:
        raw = gen_fn([{"role":"user","content":prompt}], max_new_tokens=10)
        m   = re.findall(r"\d+", raw if isinstance(raw,str) else sanitize(raw))
        if m: return min(max(float(m[0])/100.0, 0.0), 1.0)
    except Exception as e:
        print(f"Judge error: {e}")
    return 0.5


print("LLM-as-a-Judge (Qwen)...")
qwen_judge = []
for _, row in tqdm(df_qwen.iterrows(), total=len(df_qwen), desc="Qwen Judge"):
    refs = parse_refs(row.get("retrieved_references_json","[]"))
    ctx  = "\n".join([f"- {r['reference']}: {r['text_preview']}" for r in refs])
    qwen_judge.append(llm_judge(row["question"], ctx, row["answer"], row["expected_answer"], generate_qwen))

df_qwen["llm_judge_score"] = qwen_judge
df_qwen["overall_score"]   = sum(df_qwen[k].clip(0,1)*w for k,w in WEIGHTS.items())
df_qwen["quality_label"]   = pd.cut(df_qwen["overall_score"],bins=[-0.01,0.50,0.70,0.85,1.01],
                                      labels=["poor","fair","good","excellent"])
df_qwen["model_name"]       = "Qwen3.5-9B"
print("Qwen judge done. Overall score mean:", round(df_qwen["overall_score"].mean(),4))


In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

ordered_cols = [
    "id","topic","question","expected_answer","answer","overall_score","quality_label",
    "llm_judge_score","semantic_similarity","keyword_coverage","keyword_hits","keyword_missing",
    "retrieval_citation_coverage","retrieval_citation_hit","retrieval_law_hit","retrieval_article_hit",
    "answer_citation_hit","expected_article_rank","expected_law_rank","article_mrr","law_mrr",
    "precision_at_3","article_hit_at_3","article_hit_at_8","answer_word_count","answer_length_score",
    "latency_seconds","model_id","top1_reference","retrieved_references_text","retrieved_references_json",
]
ordered_cols = [c for c in ordered_cols if c in df_qwen.columns]

qwen_csv   = OUTPUT_DIR + "/evaluation_table_qwen.csv"
qwen_excel = OUTPUT_DIR + "/evaluation_report_qwen.xlsx"
df_qwen[ordered_cols].to_csv(qwen_csv, index=False, encoding="utf-8-sig")
with pd.ExcelWriter(qwen_excel, engine="openpyxl") as writer:
    df_qwen[ordered_cols].to_excel(writer, index=False, sheet_name="detail_qwen")
print("Saved:", qwen_csv)
print("Saved:", qwen_excel)


## 10. Unload Qwen → Muat GLM-4-9B-Chat

Retrieval (ChromaDB, BM25, reranker) **tetap aktif** — hanya LLM generation yang diganti.
GLM-4 akan menggunakan **retrieved docs yang sama persis** dengan Qwen untuk perbandingan yang adil.


In [ ]:
del qwen_model
del qwen_tokenizer
gc.collect()
torch.cuda.empty_cache()
print("Qwen unloaded.")
if torch.cuda.is_available():
    print(f"VRAM allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

glm4_quant = None
if torch.cuda.is_available():
    glm4_quant = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,  # bfloat16 lebih stabil untuk GLM-4
    )

# GLM-4 WAJIB trust_remote_code=True: tokenizer memakai implementasi custom
glm4_tokenizer = AutoTokenizer.from_pretrained(
    GLM4_MODEL_ID,
    trust_remote_code=True,
)

glm4_model = AutoModelForCausalLM.from_pretrained(
    GLM4_MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    quantization_config=glm4_quant,
    trust_remote_code=True,
)
glm4_model.eval()
print("GLM-4 ready:", GLM4_MODEL_ID)
if torch.cuda.is_available():
    print(f"VRAM allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")


In [ ]:
def generate_glm4(messages, max_new_tokens=MAX_NEW_TOKENS):
    """
    GLM-4 memakai apply_chat_template dengan tokenize=True, return_dict=True
    untuk langsung mendapat input_ids + attention_mask.
    Ini berbeda dari Qwen yang memakai tokenizer(prompt_text, ...).
    """
    inputs = glm4_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )
    inputs = {k: v.to(glm4_model.device) for k, v in inputs.items()}
    input_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        out = glm4_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.04,
            top_p=0.9,
        )
    return glm4_tokenizer.decode(out[0][input_len:], skip_special_tokens=True).strip()


def run_rag_glm4(question, refs_json, max_docs=8):
    """Gunakan retrieved docs yang sudah disimpan dari Qwen (perbandingan adil)."""
    t0   = time.time()
    refs = json.loads(refs_json) if isinstance(refs_json, str) else refs_json
    docs = [{"text":r["text_preview"],"metadata":{"citation_text":r["reference"]}} for r in refs]
    msgs = build_messages(question, docs, max_docs=max_docs)
    ans  = sanitize(generate_glm4(msgs))
    return {"answer":ans, "latency_seconds":time.time()-t0}

print("GLM-4 generation function ready.")


## 11. Evaluasi Batch — GLM-4-9B-Chat (Komparasi)

Menggunakan **dokumen yang sama persis** dengan Qwen (`all_docs_json`) untuk perbandingan murni generasi.


In [ ]:
print("Batch inference GLM-4...")
glm4_results = []
for row, refs_j in tqdm(zip(gt.to_dict("records"), all_docs_json), total=len(gt), desc="GLM-4 inference"):
    out = run_rag_glm4(row["question"], refs_j)
    parsed = json.loads(refs_j) if isinstance(refs_j, str) else refs_j
    glm4_results.append({
        **row,
        "model_id": GLM4_MODEL_ID,
        "answer": out["answer"],
        "latency_seconds": out["latency_seconds"],
        "retrieved_references_json": refs_j,
        "retrieved_references_text": " || ".join([r["reference"] for r in parsed]),
        "top1_reference": parsed[0]["reference"] if parsed else "",
        "retrieved_count": len(parsed),
    })
df_glm4 = pd.DataFrame(glm4_results)
print("GLM-4 inference selesai:", len(df_glm4))


In [ ]:
df_glm4 = compute_metrics(df_glm4)
df_glm4 = compute_semantic_sim(df_glm4)

print("LLM-as-a-Judge (GLM-4)...")
glm4_judge = []
for _, row in tqdm(df_glm4.iterrows(), total=len(df_glm4), desc="GLM-4 Judge"):
    refs = parse_refs(row.get("retrieved_references_json","[]"))
    ctx  = "\n".join([f"- {r['reference']}: {r['text_preview']}" for r in refs])
    glm4_judge.append(llm_judge(row["question"], ctx, row["answer"], row["expected_answer"], generate_glm4))

df_glm4["llm_judge_score"] = glm4_judge
df_glm4["overall_score"]   = sum(df_glm4[k].clip(0,1)*w for k,w in WEIGHTS.items())
df_glm4["quality_label"]   = pd.cut(df_glm4["overall_score"],bins=[-0.01,0.50,0.70,0.85,1.01],
                                      labels=["poor","fair","good","excellent"])
df_glm4["model_name"]       = "GLM-4-9B"
print("GLM-4 judge done. Overall score mean:", round(df_glm4["overall_score"].mean(),4))


## 12. Plot Evaluasi & Komparasi Model


In [ ]:
os.makedirs(PLOT_DIR, exist_ok=True)

def save_fig(name):
    path = PLOT_DIR + "/" + name
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.show(); plt.close()

metric_cols = ["llm_judge_score","semantic_similarity","keyword_coverage",
               "retrieval_citation_coverage","retrieval_law_hit","retrieval_article_hit",
               "answer_citation_hit","article_hit_at_3","article_mrr","precision_at_3","answer_length_score"]

# 01 Overall score per question
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
for ax, (df_p, lbl) in zip(axes, [(df_qwen,"Qwen3.5-9B"),(df_glm4,"GLM-4-9B")]):
    ax.bar(df_p["id"].astype(str), df_p["overall_score"]); ax.set_ylim(0,1)
    ax.set_title(f"Overall Score per Question — {lbl}"); ax.tick_params(axis="x",rotation=45)
save_fig("01_overall_score_per_question.png")

# 02 Distribution
fig, axes = plt.subplots(1,2,figsize=(14,5))
for ax,(df_p,lbl) in zip(axes,[(df_qwen,"Qwen3.5-9B"),(df_glm4,"GLM-4-9B")]):
    ax.hist(df_p["overall_score"],bins=np.linspace(0,1,11),edgecolor="black")
    ax.set_title(f"Score Distribution — {lbl}"); ax.set_xlim(0,1)
save_fig("02_overall_score_distribution.png")

# 03 Average metrics
means_q = df_qwen[metric_cols].mean().sort_values()
plt.figure(figsize=(10,6)); plt.barh(means_q.index, means_q.values); plt.xlim(0,1)
plt.title("Average Metric Scores — Qwen3.5-9B"); save_fig("03_avg_metrics_qwen.png")

# 04 Heatmap Qwen
heat_q = df_qwen.set_index("id")[metric_cols]
plt.figure(figsize=(12,7)); plt.imshow(heat_q.values,aspect="auto",vmin=0,vmax=1)
plt.xticks(range(len(metric_cols)),metric_cols,rotation=45,ha="right")
plt.yticks(range(len(heat_q)),heat_q.index); plt.colorbar(label="Score")
plt.title("Metric Heatmap — Qwen3.5-9B"); save_fig("04_heatmap_qwen.png")

# 05 Heatmap GLM-4
heat_g = df_glm4.set_index("id")[metric_cols]
plt.figure(figsize=(12,7)); plt.imshow(heat_g.values,aspect="auto",vmin=0,vmax=1)
plt.xticks(range(len(metric_cols)),metric_cols,rotation=45,ha="right")
plt.yticks(range(len(heat_g)),heat_g.index); plt.colorbar(label="Score")
plt.title("Metric Heatmap — GLM-4-9B"); save_fig("05_heatmap_glm4.png")

# 06 Latency vs score
fig, axes = plt.subplots(1,2,figsize=(14,5))
for ax,(df_p,lbl) in zip(axes,[(df_qwen,"Qwen3.5-9B"),(df_glm4,"GLM-4-9B")]):
    ax.scatter(df_p["latency_seconds"],df_p["overall_score"])
    for _,r in df_p.iterrows(): ax.text(r["latency_seconds"],r["overall_score"],str(r["id"]),fontsize=8)
    ax.set_title(f"Latency vs Score — {lbl}"); ax.set_ylim(0,1)
save_fig("06_latency_vs_score.png")

# 07 Quality labels
fig, axes = plt.subplots(1,2,figsize=(12,5))
for ax,(df_p,lbl) in zip(axes,[(df_qwen,"Qwen3.5-9B"),(df_glm4,"GLM-4-9B")]):
    cnt = df_p["quality_label"].value_counts().reindex(["poor","fair","good","excellent"]).fillna(0)
    ax.bar(cnt.index.astype(str),cnt.values); ax.set_title(f"Quality Label — {lbl}")
save_fig("07_quality_labels.png")

# 08 Model comparison bar
compare_cols = ["overall_score","llm_judge_score","semantic_similarity",
                "keyword_coverage","retrieval_citation_coverage",
                "retrieval_article_hit","answer_citation_hit","article_mrr","answer_length_score"]
q_m = df_qwen[compare_cols].mean(); g_m = df_glm4[compare_cols].mean()
compare_df = pd.concat([q_m.rename("Qwen3.5-9B"), g_m.rename("GLM-4-9B")], axis=1)
compare_df["diff"] = (compare_df["GLM-4-9B"]-compare_df["Qwen3.5-9B"]).round(4)
fig, ax = plt.subplots(figsize=(13,6))
x, w = list(range(len(compare_cols))), 0.35
ax.bar([i-w/2 for i in x], compare_df["Qwen3.5-9B"].clip(0,1), w, label="Qwen3.5-9B")
ax.bar([i+w/2 for i in x], compare_df["GLM-4-9B"].clip(0,1), w, label="GLM-4-9B")
ax.set_xticks(x); ax.set_xticklabels(compare_cols,rotation=40,ha="right")
ax.set_ylim(0,1.1); ax.set_title("Komparasi Metrik: Qwen3.5-9B vs GLM-4-9B-Chat")
ax.set_ylabel("Mean Score"); ax.legend()
save_fig("08_model_comparison.png")

print("\n=== TABEL KOMPARASI ===")
print(compare_df.round(4).to_string())


In [ ]:
# Simpan semua hasil
compare_df.to_csv(OUTPUT_DIR+"/model_comparison.csv", encoding="utf-8-sig")
df_glm4[[c for c in ordered_cols if c in df_glm4.columns]].to_csv(
    OUTPUT_DIR+"/evaluation_table_glm4.csv", index=False, encoding="utf-8-sig")

# Combined Excel dengan kedua model
with pd.ExcelWriter(OUTPUT_DIR+"/evaluation_combined.xlsx", engine="openpyxl") as writer:
    df_qwen[[c for c in ordered_cols if c in df_qwen.columns]].to_excel(writer,index=False,sheet_name="Qwen3.5-9B")
    df_glm4[[c for c in ordered_cols if c in df_glm4.columns]].to_excel(writer,index=False,sheet_name="GLM-4-9B")
    compare_df.to_excel(writer, sheet_name="Komparasi")

print("Semua hasil tersimpan di:", OUTPUT_DIR)
print(f"Qwen overall  : {df_qwen['overall_score'].mean():.4f}")
print(f"GLM-4 overall : {df_glm4['overall_score'].mean():.4f}")
